<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/fabiobento/rl-course-hf/blob/main/unit1/unit1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/fabiobento/rl-course-hf/blob/main/unit1/unit1.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

Adaptado do [repositório](https://github.com/huggingface/deep-rl-class) do [Hugging Face Deep Reinforcement Learning Course](https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt)

# Unidade 1: Treine seu primeiro agente de aprendizado por reforço profundo 🤖

Neste notebook, você treinará seu **primeiro agente de aprendizado por reforço profundo**(_Deep Reinforcement Learning agent_-Deep RL): um agente Lunar Lander que aprenderá a **pousar corretamente na Lua 🌕**.

Você usatá  [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/), uma biblioteca de Deep RL.

In [23]:
%%html
<video controls autoplay><source src="https://huggingface.co/sb3/ppo-LunarLander-v2/resolve/main/replay.mp4" type="video/mp4"></video>

### O ambiente 🎮

- [LunarLander-v2](https://gymnasium.farama.org/environments/box2d/lunar_lander/)

### A biblioteca utilizada 📚

- [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/)

## Uma breve recapitulação sobre Deep RL 📚

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/RL_process_game.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">
Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Vamos fazer uma pequena recapitulação do que aprendemos na primeira unidade:

- O aprendizado por reforço é uma **abordagem computacional para aprender com ações**. Criamos um agente que aprende com o ambiente **interagindo com ele por meio de tentativa e erro** e recebendo recompensas (negativas ou positivas) como feedback.

- O objetivo de qualquer agente RL é **maximizar sua recompensa cumulativa esperada** (também chamada de retorno esperado), porque o RL se baseia na _hipótese da recompensa_, que é que todos os objetivos podem ser descritos como a maximização de uma recompensa cumulativa esperada.

- O processo RL é um **loop que gera uma sequência de estado, ação, recompensa e próximo estado**.

- Para calcular a recompensa cumulativa esperada (retorno esperado), **descontamos as recompensas**: as recompensas que vêm mais cedo (no início do jogo) são mais prováveis de acontecer, pois são mais previsíveis do que a recompensa futura a longo prazo.

- Para resolver um problema de RL, você deseja **encontrar uma política ideal**; a política é o “cérebro” da sua IA que nos dirá qual ação tomar em um determinado estado. A ideal é aquela que fornece as ações que maximizam o retorno esperado.

- Existem **duas** maneiras de encontrar sua política ideal(_optimal policy_):

> 1. **Treinando sua política diretamente**: métodos baseados em políticas(_policy-based methods_).
> 2. **Treinando uma função de valor** que nos diz o retorno esperado que o agente obterá em cada estado e usando essa função para definir nossa política: métodos baseados em valor(_value based methods_).

- Por fim, falamos sobre RL profundo porque **introduzimos redes neurais profundas para estimar a ação a ser tomada (_policy-based) ou para estimar o valor de um estado (_value based), daí o nome “profundo”**.

## Instalar dependências 🔽

The first step is to install the dependencies, we’ll install multiple ones.

- `swig`: O `swig` é necessário porque alguns ambientes do Gymnasium (como o LunarLander-v3, que usa Box2D) dependem de bibliotecas escritas em C/C++. O `swig` é uma ferramenta que gera automaticamente as interfaces entre essas bibliotecas em C/C++ e o Python, permitindo que o código Python utilize funcionalidades dessas bibliotecas nativas.
- `gymnasium[box2d]`: Contém o ambiente LunarLander-v3 🌛
- `stable-baselines3[extra]`: A biblioteca de _deep reinforcement learning_.

Pra facilitar criamos um sript para instalar as dependências.

In [24]:
%pip install swig
%pip install gymnasium[box2d]
%pip install stable-baselines3[extra]

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Importar as bibliotecas 📦

In [25]:

import gymnasium
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

## Entenda o Gymnasium e como ele funciona 🤖

🏋 A biblioteca contendo o nosso ambiente é chamada Gymnasium.
**Você utilizará bastante o Gymnasium no Deep Reinforcement Learning(Deep RL).**

Gymnasium é a **nova versão da biblioteca Gym** [mantida pela Farama Foundation](https://farama.org/).

A biblioteca do Gymnasium oferece duas coisas:

- Uma interface que permite **criar ambientes RL**.
- Uma **coleção de ambientes** (gym-control, atari, box2D...).

Vejamos um exemplo, mas primeiro vamos relembrar o loop RL.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/RL_process_game.jpg" alt="The RL process" width="100%">

Em cada etapa:
- Nosso agente recebe um **state (S0)** do **Environment** — recebemos o primeiro frame do nosso jogo (_environment_).
- Com base nesse **state (S0)**, o agente realiza uma **action (A0)** — nosso agente se moverá para a direita.
- O ambiente transita para um **novo** **state (S1)** — novo frame.
- O ambiente dá alguma **reward (R1)** ao agente — não estamos mortos *(Recompensa positiva +1)*.



Com Gymnasium:

1️⃣ Nós criamos um ambiente usando `gymnasium.make()`

2️⃣ Resetamos o ambiente para seu estado inicial com `observation = env.reset()`

A cada etapa:

3️⃣ Obtenha uma ação usando nosso modelo (em nosso exemplo, tomamos uma ação aleatória)

4️⃣ Usando `env.step(action)`, executamos essa ação no ambiente e recebemos
- `observation`: O novo estado (st+1)
- `reward`: A recompensa que recebemos apos a execução da ação
- `terminated`: Indica se o episódio foi encerrado (o agente atingiu o _terminal state_)
- `truncated`: Introduzido com esta nova versão, indica um limite de tempo ou se um agente sai dos limites do ambiente, por exemplo.
- `info`: Um dicionário que fornece informações adicionais (depende do ambiente).

Para mais explicações confira aqui 👉 https://gymnasium.farama.org/api/env/#gymnasium.Env.step

Se o episódio for encerrado:
- Reiniciamos o ambiente para seu estado inicial com `observation = env.reset()`

**Vamos ver um exemplo!** Certifique-se de ler o código

In [26]:
import gymnasium as gym

# Primeiro, criamos nosso ambiente chamado LunarLander-v2
env = gym.make("LunarLander-v3")

# Em seguida, resetamos esse ambiente
observation, info = env.reset()

for _ in range(20):
  # Executa uma ação aleatória
  action = env.action_space.sample()
  print("Ação executada:", action)

  # Realiza essa ação no ambiente e obtém
  # próximo_estado, recompensa, terminado, truncado e info
  observation, reward, terminated, truncated, info = env.step(action)

  # Se o jogo terminou (pousou, colidiu) ou foi truncado (tempo esgotado)
  if terminated or truncated:
      # Reseta o ambiente
      print("Ambiente foi resetado")
      observation, info = env.reset()

env.close()


Ação executada: 3
Ação executada: 2
Ação executada: 1
Ação executada: 0
Ação executada: 2
Ação executada: 3
Ação executada: 1
Ação executada: 2
Ação executada: 0
Ação executada: 1
Ação executada: 1
Ação executada: 3
Ação executada: 2
Ação executada: 1
Ação executada: 3
Ação executada: 1
Ação executada: 3
Ação executada: 2
Ação executada: 3
Ação executada: 3


## Criar o ambiente LunarLander 🌛 E entender como ele funciona

### [O ambiente 🎮](https://gymnasium.farama.org/environments/box2d/lunar_lander/)

Neste primeiro tutorial, vamos treinar nosso agente, um [Lunar Lander](https://gymnasium.farama.org/environments/box2d/lunar_lander/), **para pousar corretamente na lua**. Para isso, o agente precisa aprender **a adaptar sua velocidade e posição (horizontal, vertical e angular) para pousar corretamente.**

---


💡 Um bom hábito quando você começa a usar um ambiente é verificar sua documentação.

👉 https://gymnasium.farama.org/environments/box2d/lunar_lander/

---


Vamos ver como o ambiente se parece:


In [27]:
# Criamos nosso ambiente com gym.make("<nome_do_ambiente>")
env = gym.make("LunarLander-v3")
env.reset()
print("_____OBSERVATION SPACE_____ \n")
print("Dimensões do Observation Space", env.observation_space.shape)
print("Observação amostrada aleatoriamente", env.observation_space.sample())

_____OBSERVATION SPACE_____ 

Dimensões do Observation Space (8,)
Observação amostrada aleatoriamente [ 1.282493    1.1664083  -2.2841125   4.5110583   4.057352    0.5514445
  0.66471267  0.33841276]


Vemos com `Observation Space Shape (8,)` que a observação é um vetor de tamanho 8, onde cada valor contém informações diferentes sobre o módulo de pouso:
- Coordenada horizontal da plataforma (x)
- Coordenada vertical da plataforma (y)
- Velocidade horizontal (x)
- Velocidade vertical (y)
- Ângulo
- Velocidade angular
- Se o ponto de contato da perna esquerda tocou o solo (booleano)
- Se o ponto de contato da perna direita tocou o solo (booleano)


In [28]:
print("\n _____ACTION SPACE_____ \n")
print("Formato do Action Space", env.action_space.n)
print("Action Space amostrada aleatoriamente", env.action_space.sample()) # Executar uma ação randômica


 _____ACTION SPACE_____ 

Formato do Action Space 4
Action Space amostrada aleatoriamente 1


O action space (o conjunto de ações possíveis que o agente pode realizar) é discreto, com 4 ações disponíveis. 🎮:

- Action 0: Não fazer nada,
- Action 1: Acionar o motor de orientação esquerdo,
- Action 2: Acionar o motor principal,
- Action 3: Acionar o motor de orientação direito.

Reward function (a função que dará uma recompensa a cada intervalo de tempo) 💰:

Após cada etapa, é concedida uma recompensa. A recompensa total de um episódio é a **soma das recompensas de todas as etapas desse episódio**.

Para cada etapa, a recompensa:

- É aumentada/diminuída quanto mais próximo/distante o módulo de pouso estiver da plataforma de pouso.
- É aumentada/diminuída quanto mais lento/rápido o módulo de pouso estiver se movendo.
- É diminuída quanto mais o módulo de pouso estiver inclinado (ângulo não horizontal).
- É aumentada em 10 pontos para cada perna que estiver em contato com o solo.
- É diminuída em 0,03 pontos a cada quadro em que um motor lateral estiver funcionando.
- É diminuída em 0,3 pontos a cada quadro em que o motor principal estiver funcionando.

O episódio recebe uma **recompensa adicional de -100 ou +100 pontos por colidir ou pousar com segurança, respectivamente.**

Um episódio é **considerado uma solução se obtiver pelo menos 200 pontos.**

#### Ambiente Vetorizado (_Vectorized Environment_)

- Criamos um ambiente vetorizado (um método para empilhar vários ambientes independentes em um único ambiente) de 16 ambientes, dessa forma, **teremos experiências mais diversificadas durante o treinamento.**

In [ ]:
# Cria o ambiente vetorizado com 16 ambientes
# Isso é útil para treinar o agente em múltiplos ambientes simultaneamente, aumentando a
# diversidade das experiências de treinamento e acelerando o processo de aprendizado.
env = make_vec_env('LunarLander-v3', n_envs=16)

## Criar o Modelo 🤖
- We have studied our environment and we understood the problem: **being able to land the Lunar Lander to the Landing Pad correctly by controlling left, right and main orientation engine**. Now let's build the algorithm we're going to use to solve this Problem 🚀.

- To do so, we're going to use our first Deep RL library, [Stable Baselines3 (SB3)](https://stable-baselines3.readthedocs.io/en/master/).

- SB3 is a set of **reliable implementations of reinforcement learning algorithms in PyTorch**.

---

💡 A good habit when using a new library is to dive first on the documentation: https://stable-baselines3.readthedocs.io/en/master/ and then try some tutorials.

----

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/sb3.png" alt="Stable Baselines3">

To solve this problem, we're going to use SB3 **PPO**. [PPO (aka Proximal Policy Optimization) is one of the SOTA (state of the art) Deep Reinforcement Learning algorithms that you'll study during this course](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html#example%5D).

PPO is a combination of:
- *Value-based reinforcement learning method*: learning an action-value function that will tell us the **most valuable action to take given a state and action**.
- *Policy-based reinforcement learning method*: learning a policy that will **give us a probability distribution over actions**.

Stable-Baselines3 is easy to set up:

1️⃣ You **create your environment** (in our case it was done above)

2️⃣ You define the **model you want to use and instantiate this model** `model = PPO("MlpPolicy")`

3️⃣ You **train the agent** with `model.learn` and define the number of training timesteps

```
# Create environment
env = gym.make('LunarLander-v2')

# Instantiate the agent
model = PPO('MlpPolicy', env, verbose=1)
# Train the agent
model.learn(total_timesteps=int(2e5))
```



In [ ]:
# SOLUTION
# We added some parameters to accelerate the training
model = PPO(
    policy = 'MlpPolicy',
    env = env,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.999,
    gae_lambda = 0.98,
    ent_coef = 0.01,
    verbose=1)

## Train the PPO agent 🏃
- Let's train our agent for 1,000,000 timesteps, don't forget to use GPU on Colab. It will take approximately ~20min, but you can use fewer timesteps if you just want to try it out.
- During the training, take a ☕ break you deserved it 🤗

#### Solution

In [9]:
# SOLUTION
# Train it for 1,000,000 timesteps
model.learn(total_timesteps=1000000)
# Save the model
model_name = "ppo-LunarLander-v2"
model.save(model_name)

KeyboardInterrupt: 

## Evaluate the agent 📈
- Remember to wrap the environment in a [Monitor](https://stable-baselines3.readthedocs.io/en/master/common/monitor.html).
- Now that our Lunar Lander agent is trained 🚀, we need to **check its performance**.
- Stable-Baselines3 provides a method to do that: `evaluate_policy`.
- To fill that part you need to [check the documentation](https://stable-baselines3.readthedocs.io/en/master/guide/examples.html#basic-usage-training-saving-loading)
- In the next step,  we'll see **how to automatically evaluate and share your agent to compete in a leaderboard, but for now let's do it ourselves**


💡 When you evaluate your agent, you should not use your training environment but create an evaluation environment.

In [ ]:
#@title
eval_env = Monitor(gym.make("LunarLander-v3", render_mode='rgb_array'))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

In [ ]:
from matplotlib import animation
from IPython.display import HTML

# Gera um vídeo do agente treinado no ambiente de avaliação
import matplotlib.pyplot as plt

frames = []
obs = eval_env.reset()[0]
done = False

while not done:
    # O modelo prevê a ação
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    frame = eval_env.render()
    frames.append(frame)
    done = terminated or truncated

eval_env.close()

# Função para animar os frames
def display_video(frames):
    fig = plt.figure(figsize=(6, 6))
    plt.axis('off')
    im = plt.imshow(frames[0])

    def animate(i):
        im.set_array(frames[i])
        return [im]

    anim = animation.FuncAnimation(fig, animate, frames=len(frames), interval=50, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

display_video(frames)

## Some additional challenges 🏆
The best way to learn **is to try things by your own**! As you saw, the current agent is not doing great. As a first suggestion, you can train for more steps. With 1,000,000 steps, we saw some great results!

In the [Leaderboard](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard) you will find your agents. Can you get to the top?

Here are some ideas to achieve so:
* Train more steps
* Try different hyperparameters for `PPO`. You can see them at https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html#parameters.
* Check the [Stable-Baselines3 documentation](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html) and try another model such as DQN.
* **Push your new trained model** on the Hub 🔥

**Compare the results of your LunarLander-v2 with your classmates** using the [leaderboard](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard) 🏆

Is moon landing too boring for you? Try to **change the environment**, why not use MountainCar-v0, CartPole-v1 or CarRacing-v0? Check how they work [using the gym documentation](https://www.gymlibrary.dev/) and have fun 🎉.